# EL-GHALI MOHAMED


### RANDOM FOREST

In [4]:
import pandas as pd
import math
import random    # Pour le tirage aléatoire
from collections import Counter

# Création du dataset
data = {
    'Perspectives': ['Ensoleille', 'Ensoleille', 'Couvert', 'Pluie', 'Pluie', 'Pluie', 'Couvert',
                     'Ensoleille', 'Ensoleille', 'Pluie', 'Ensoleille', 'Couvert', 'Couvert', 'Pluie'],
    'Temperature': ['Chaude', 'Chaude', 'Chaude', 'Douce', 'Fraiche', 'Fraiche', 'Fraiche',
                    'Douce', 'Fraiche', 'Douce', 'Douce', 'Douce', 'Chaude', 'Douce'],
    'Humidite': ['Haute', 'Haute', 'Haute', 'Haute', 'Normale', 'Normale', 'Normale',
                 'Haute', 'Normale', 'Normale', 'Normale', 'Haute', 'Normale', 'Haute'],
    'Vent': ['Faible', 'Fort', 'Faible', 'Faible', 'Faible', 'Fort', 'Fort',
             'Faible', 'Faible', 'Faible', 'Fort', 'Fort', 'Faible', 'Fort'],
    'Jouer': ['Non', 'Non', 'Oui', 'Oui', 'Oui', 'Non', 'Oui',
              'Non', 'Oui', 'Oui', 'Oui', 'Oui', 'Oui', 'Non']
}
df = pd.DataFrame(data)

### LES FONCTION MATHEMATIQUES

1. **L'Entropie ($H$)** : Elle mesure l'impureté ou le désordre dans le jeu de données. Plus les données sont mélangées (ex: 50% Oui, 50% Non), plus l'entropie est proche de 1.

$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$


2. **Le Gain d'Information ($Gain$)** : Il mesure la réduction de l'entropie après avoir divisé les données selon un attribut spécifique. L'ID3 choisit toujours l'attribut qui maximise ce gain.

$$Gain(S, A) = H(S) - \sum_{v \in Valeurs(A)} \frac{|S_v|}{|S|} H(S_v)$$


In [5]:
def calcul_entropie(colonne_cible):

    # On compte combien de fois apparaît chaque classe
    occurrences = colonne_cible.value_counts().tolist()
    total = sum(occurrences)

    entropie = 0
    # Application de la formule : - somme(p * log2(p))
    for occ in occurrences:
        probabilite = occ / total
        entropie -= probabilite * math.log2(probabilite)

    return entropie

def gain_information(donnees, attribut_split, nom_cible):
    # 1. Calcul du désordre AVANT la division
    entropie_totale = calcul_entropie(donnees[nom_cible])

    # 2. Calcul du désordre APRÈS la division (Entropie pondérée des sous-groupes)
    valeurs = donnees[attribut_split].unique() # Ex: ['Chaude', 'Douce', 'Fraiche']
    entropie_ponderee = 0
    total_lignes = len(donnees)

    for val in valeurs:
        # On isole les lignes correspondant à chaque valeur
        sous_ensemble = donnees[donnees[attribut_split] == val]
        # On calcule le poids de ce sous-groupe par rapport au total
        probabilite_sous_ensemble = len(sous_ensemble) / total_lignes
        # On ajoute l'entropie de ce sous-groupe multipliée par son poids
        entropie_ponderee += probabilite_sous_ensemble * calcul_entropie(sous_ensemble[nom_cible])

    # Le gain est la différence entre le désordre de départ et le désordre d'arrivée
    return entropie_totale - entropie_ponderee

### Construction d'un Arbre Aléatoire

In [6]:
def construire_arbre_aleatoire(donnees, dataset_original, colonnes_features, nom_cible, nb_features_max, classe_parent=None):


    # Création des feuilles de l'arbre

    # Cas 1 : Toutes les lignes restantes ont la même réponse
    if len(donnees[nom_cible].unique()) <= 1:
        return donnees[nom_cible].iloc[0]

    # Cas 2 : Il n'y a plus de données à analyser dans cette branche
    elif len(donnees) == 0:
        return dataset_original[nom_cible].value_counts().idxmax()

    # Cas 3 : On a utilisé toutes les caractéristiques possibles
    elif len(colonnes_features) == 0:
        return donnees[nom_cible].value_counts().idxmax()

    # CONSTRUCTION DU NŒUD
    else:
        # On garde en mémoire la classe majoritaire au cas où les branches suivantes seraient vides
        classe_parent = donnees[nom_cible].value_counts().idxmax()

        # Sous-échantillonnage des features
        # On détermine combien de features on va tester au maximum
        nb_features_a_tester = min(nb_features_max, len(colonnes_features))
        # On tire au hasard les features parmi celles disponibles (sans remise)
        features_aleatoires = random.sample(colonnes_features, nb_features_a_tester)

        # On évalue le Gain d'Information UNIQUEMENT sur ces features tirées au sort
        gains = [gain_information(donnees, feature, nom_cible) for feature in features_aleatoires]

        # On identifie la feature gagnante celle avec le gain le plus élevé
        index_meilleur_feature = gains.index(max(gains))
        meilleur_feature = features_aleatoires[index_meilleur_feature]

        # Initialisation du nœud de l'arbre
        arbre = {meilleur_feature: {}}

        # On retire la feature gagnante de la liste pour les prochains étages de l'arbre
        features_restantes = [f for f in colonnes_features if f != meilleur_feature]

        # Pour chaque valeur possible de cette meilleure feature, on crée une branche
        for valeur in donnees[meilleur_feature].unique():
            # On filtre les données pour cette branche spécifique
            sous_donnees = donnees[donnees[meilleur_feature] == valeur]

            # On relance la fonction pour construire l'étage suivant
            sous_arbre = construire_arbre_aleatoire(sous_donnees, dataset_original, features_restantes, nom_cible, nb_features_max, classe_parent)

            # On attache le sous-arbre au nœud actuel
            arbre[meilleur_feature][valeur] = sous_arbre

        return arbre

def predire_arbre(requete, arbre):
    """Navigue de haut en bas dans un seul arbre pour faire une prédiction."""
    for cle in list(requete.keys()):
        if cle in list(arbre.keys()):
            try:
                # On descend dans la branche correspondant à la valeur de notre requête
                resultat = arbre[cle][requete[cle]]
            except KeyError:
                # Si l'arbre n'a jamais vu cette combinaison précise lors de l'entraînement
                return "Inconnu"

            # Si le résultat est un dict, on n'est pas encore à une feuille, on continue la récursivité
            if isinstance(resultat, dict):
                return predire_arbre(requete, resultat)
            else:
                return resultat # On a atteint une feuille, on retourne la prédiction

### Construction et Vote de la Forêt (Bagging)
Nous combinons l'échantillonnage des données (Bootstrap) et la création de multiples arbres pour former la Forêt.

In [7]:
def creer_echantillon_bootstrap(dataframe):
    """
    Le Bagging (Bootstrap Aggregating) :
    Crée un nouveau dataset en tirant des lignes au hasard *avec remise*.
    Cela signifie qu'une même ligne peut apparaître plusieurs fois, et d'autres jamais.
    Cela aide à réduire le surapprentissage (overfitting).
    """
    indices_aleatoires = [random.randint(0, len(dataframe) - 1) for _ in range(len(dataframe))]
    return dataframe.iloc[indices_aleatoires]

def entrainer_random_forest(donnees, features, cible, nb_arbres=10):
    foret = []

    # Random Forest (Classification) :
    # À chaque nœud, on regarde la racine carrée du nombre total de features.
    # Ici on a 4 features, donc math.sqrt(4) = 2 features testées par nœud.
    nb_features_max = max(1, int(math.sqrt(len(features))))

    for i in range(nb_arbres):
        # Création d'un jeu de données unique pour cet arbre
        echantillon = creer_echantillon_bootstrap(donnees)

        # Construction de l'arbre avec ce jeu de données
        arbre = construire_arbre_aleatoire(echantillon, donnees, features, cible, nb_features_max)

        # Ajout de l'arbre à notre collection
        foret.append(arbre)

    print(f"Forêt entraînée avec succès ! ({nb_arbres} arbres plantés)")
    return foret

def predire_random_forest(requete, foret):
    """
    Demande à chaque arbre sa prédiction, puis organise un vote majoritaire
    pour rendre la décision finale .
    """
    votes = []

    # Interrogation individuelle de chaque arbre
    for arbre in foret:
        prediction = predire_arbre(requete, arbre)
        # On exclut les arbres qui n'ont pas su répondre à cause de données inconnues
        if prediction != "Inconnu":
            votes.append(prediction)

    if not votes:
        return "Impossible de prédire", {}

    # Comptage des votes
    compteur_votes = Counter(votes)
    # Récupération du choix ayant reçu le plus de votes
    prediction_finale = compteur_votes.most_common(1)[0][0]

    return prediction_finale, dict(compteur_votes)

### Exécution et Tests
On définit les colonnes à utiliser et on lance l'entraînement de la Forêt.

In [8]:
# Définition de ce qui est en entrée (features) et en sortie (cible)
colonnes_features = ['Perspectives', 'Temperature', 'Humidite', 'Vent']
colonne_cible = 'Jouer'

# On décide de planter 15 arbres pour notre forêt
ma_foret = entrainer_random_forest(df, colonnes_features, colonne_cible, nb_arbres=15)

# On crée une nouvelle météo (notre "requete")
jour_test = {'Perspectives': 'Ensoleille', 'Temperature': 'Fraiche', 'Humidite': 'Normale', 'Vent': 'Faible'}

# On demande à la forêt de faire une prédiction
prediction, details_votes = predire_random_forest(jour_test, ma_foret)

print("\n=== RÉSULTATS DE LA PRÉDICTION ===")
print(f"Météo du jour : {jour_test}")
print(f"Détail des votes des arbres : {details_votes}")
print(f"-> DÉCISION DE LA FORÊT : Jouer = {prediction}")

Forêt entraînée avec succès ! (15 arbres plantés)

=== RÉSULTATS DE LA PRÉDICTION ===
Météo du jour : {'Perspectives': 'Ensoleille', 'Temperature': 'Fraiche', 'Humidite': 'Normale', 'Vent': 'Faible'}
Détail des votes des arbres : {'Oui': 11, 'Non': 3}
-> DÉCISION DE LA FORÊT : Jouer = Oui
